In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler

In [ ]:
solTestX = pd.read_csv('solTestX.txt', sep=r'\s+', engine='python')
solTrainX = pd.read_csv('solTrainX.txt', sep=r'\s+', engine='python')
solTestY = pd.read_csv('solTestY.txt', sep=r'\s+', engine='python')
solTrainY = pd.read_csv('solTrainY.txt', sep=r'\s+', engine='python')

In [ ]:
solTestX.shape, solTrainX.shape, solTestY.shape, solTrainY.shape

In [ ]:
 #, solTrainX.columns#, solTestY.head, solTrainY.head
solTrainY.columns = ['Solubility']

In [ ]:
# print("--- solTrainX Head ---")
# print(solTrainX.head())
# print("\n--- solTrainX Shape ---")
# print(solTrainX.shape)

In [ ]:
skew(solTrainY['Solubility'])

In [ ]:
plt.figure(figsize=(8,10))
sns.histplot(solTrainY, kde=True)
plt.show()
solTrainY.min(), solTrainY.max()

In [ ]:
continuous_var = [name for name in solTrainX.columns if "FP" not in name]
continuous_var
continuous_skew = solTrainX[continuous_var].apply(lambda x: skew(x.dropna())) 
top_skew = continuous_skew[continuous_skew > 1.0].sort_values(ascending=False).head(5)
top_skew

In [ ]:
x_transformed = solTrainX.copy()

transformer_yj = PowerTransformer(method='yeo-johnson')
x_continuous_transformed = transformer_yj.fit_transform(solTrainX[continuous_var])

x_continuous_transformed_df = pd.DataFrame(x_continuous_transformed, columns=continuous_var, index=solTrainX.index)
x_transformed[continuous_var] = x_continuous_transformed_df

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_transformed)
solTrainX_final = pd.DataFrame(x_scaled, columns=solTrainX.columns, index=solTrainX.index)

In [ ]:
#Pre-processamento do conjunto de teste 
x_test_transformed = solTestX.copy()
x_test_continuous_transformed = transformer_yj.transform(solTestX[continuous_var])

x_test_continuous_transformed_df = pd.DataFrame(x_test_continuous_transformed, columns=continuous_var, index=solTestX.index)
x_test_transformed[continuous_var] = x_test_continuous_transformed_df

x_test_scaled = scaler.transform(x_test_transformed)
solTestX_final = pd.DataFrame(x_test_scaled, columns=solTestX.columns, index=solTestX.index)

In [ ]:
correlation_matrix = solTrainX_final.corr()
np.fill_diagonal(correlation_matrix.values, np.nan)
max_corr = correlation_matrix.abs().max().max()

In [ ]:
predictor_result_correlation = solTrainX_final.corrwith(solTrainY['Solubility']).abs().sort_values(ascending=False)
print(f'preditores mais correlacionados com resultado: {predictor_result_correlation.head(5)}')